# 🔭 Gravitational Lensing Dark Matter Classification

A PyTorch pipeline for classifying gravitational lensing images into three categories:
- **no_substructure** (0)
- **subhalo** (1)
- **vortex** (2)

Based on:
- [Varma et al. (2020)](https://arxiv.org/pdf/2005.05353) — CNN sensitivity to sub-halo mass perturbations
- [Alexander et al. (2020)](https://doi.org/10.3847/1538-4357/ab7925) — ResNet-18 for deep learning morphologies

---

## 1 │ Setup & Imports

In [ ]:
import glob
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.notebook import tqdm

# ---- Reproducibility ----
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}  ({torch.cuda.device_count()} GPU(s))")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i} : {torch.cuda.get_device_name(i)}")

## 2 │ Configuration

In [ ]:
# ======== Hyper-parameters (edit these) ========
EPOCHS      = 50
BATCH_SIZE  = 64
LR          = 1e-4
IMG_SIZE    = 224        # resize to this square size
NUM_WORKERS = 4

# ======== Paths ========
DATA_ROOT  = "/kaggle/working/dataset"    # contains train/ and val/
TRAIN_DIR  = os.path.join(DATA_ROOT, "train")
VAL_DIR    = os.path.join(DATA_ROOT, "val")
OUTPUT_DIR = "/kaggle/working"

# ======== Class mapping (actual folder names in the zip) ========
FOLDER_TO_LABEL = {
    "no":     0,   # No Substructure
    "sphere": 1,   # Subhalo
    "vort":   2,   # Vortex
}
CLASS_NAMES = ["no_substructure", "subhalo", "vortex"]  # display names

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Train dir  : {TRAIN_DIR}")
print(f"Val dir    : {VAL_DIR}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"Class map  : {FOLDER_TO_LABEL}")

## 3 │ Download & Extract Dataset from Google Drive

In [ ]:
!pip install -q gdown

In [ ]:
import gdown

FILE_ID  = "1gF1bW9UFIodcF87ZoQENPisWxbXD0yuV"
ZIP_PATH = "/kaggle/working/dataset.zip"

# ---- Download ----
if not os.path.exists(ZIP_PATH):
    print("Downloading dataset from Google Drive...")
    !gdown --id {FILE_ID} -O {ZIP_PATH}
    print("Download complete!")
else:
    print(f"[INFO] {ZIP_PATH} already exists — skipping download.")

# ---- Extract ----
if not os.path.exists(TRAIN_DIR):
    print(f"Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/kaggle/working/")
    print("Extraction complete!")
else:
    print(f"[INFO] Data already extracted at {DATA_ROOT}")

# ---- Verify ----
print("\n--- Verifying directory structure ---")
print(f"Contents of {TRAIN_DIR}:")
print(os.listdir(TRAIN_DIR))
print(f"\nContents of {VAL_DIR}:")
print(os.listdir(VAL_DIR))

## 4 │ Data Pipeline & `LensingDataset`

Folder-to-class mapping: `no/` → 0, `sphere/` → 1, `vort/` → 2

In [ ]:
class LensingDataset(Dataset):
    def __init__(self, root, folder_to_label, transform=None):
        self.root = os.path.abspath(root)
        self.folder_to_label = folder_to_label
        self.transform = transform
        self.samples = []

        for folder_name, label in self.folder_to_label.items():
            folder_path = os.path.join(self.root, folder_name)
            if not os.path.isdir(folder_path):
                print(f"[WARNING] Folder not found: {folder_path}")
                print(f"          Available: {os.listdir(self.root)}")
                continue
            for ext in ("*.png", "*.jpg", "*.jpeg", "*.npy"):
                found = glob.glob(os.path.join(folder_path, ext))
                self.samples.extend([(os.path.abspath(f), label) for f in found])

        self.samples.sort(key=lambda x: x[0])

        if len(self.samples) == 0:
            print(f"\n[ERROR] No images found under: {self.root}")
            print(f"Root contents: {os.listdir(self.root)}")
            for item in os.listdir(self.root):
                sub = os.path.join(self.root, item)
                if os.path.isdir(sub):
                    print(f"  {item}/  →  {os.listdir(sub)[:10]}")
            raise RuntimeError(f"No images found under {self.root}")

        from collections import Counter
        counts = Counter(label for _, label in self.samples)
        print(f"[LensingDataset] Loaded {len(self.samples)} images from {self.root}")
        for fn, lbl in self.folder_to_label.items():
            print(f"  {fn}/ (label={lbl}) : {counts.get(lbl, 0)} images")

        # Debug: print shape of first sample
        first_path = self.samples[0][0]
        if first_path.endswith(".npy"):
            s = np.load(first_path).shape
        else:
            s = np.array(Image.open(first_path)).shape
        print(f"  Sample shape (raw): {s}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        # --- Load ---
        if path.endswith(".npy"):
            img = np.load(path).astype(np.float32)
        else:
            img = np.array(Image.open(path).convert("L"), dtype=np.float32)

        # --- Min-max normalise to [0, 1] ---
        img_min, img_max = img.min(), img.max()
        if img_max - img_min > 0:
            img = (img - img_min) / (img_max - img_min)

        # --- Ensure shape is (3, H, W) for ResNet ---
        # Handle all possible input shapes:
        #   (H, W)       -> grayscale 2D
        #   (H, W, 1)    -> grayscale channels-last
        #   (H, W, 3)    -> RGB channels-last
        #   (1, H, W)    -> grayscale channels-first
        #   (3, H, W)    -> RGB channels-first
        if img.ndim == 2:
            # (H, W) -> (3, H, W)
            img = np.stack([img, img, img], axis=0)
        elif img.ndim == 3:
            # If channels-last, transpose to channels-first
            if img.shape[-1] in (1, 3):
                img = np.transpose(img, (2, 0, 1))  # -> (C, H, W)
            # Now always channels-first; repeat if single channel
            if img.shape[0] == 1:
                img = np.repeat(img, 3, axis=0)     # -> (3, H, W)

        img = torch.from_numpy(img)  # (3, H, W) float32
        if self.transform:
            img = self.transform(img)
        return img, label

### Transforms & DataLoaders

In [ ]:
def get_transforms(train=True):
    t = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
    if train:
        t += [
            transforms.RandomChoice([
                transforms.RandomRotation((0, 0)),
                transforms.RandomRotation((90, 90)),
                transforms.RandomRotation((180, 180)),
                transforms.RandomRotation((270, 270)),
            ]),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
        ]
    return transforms.Compose(t)


train_dataset = LensingDataset(TRAIN_DIR, FOLDER_TO_LABEL, get_transforms(True))
val_dataset   = LensingDataset(VAL_DIR,   FOLDER_TO_LABEL, get_transforms(False))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Quick sanity check on first batch
sample_img, sample_lbl = train_dataset[0]
print(f"\nSample tensor shape : {sample_img.shape}")
print(f"Sample label        : {sample_lbl}")
print(f"Train: {len(train_dataset)}  |  Val: {len(val_dataset)}  |  Batch: {BATCH_SIZE}")

## 5 │ Model (ResNet-18)

In [ ]:
def build_model(num_classes=3):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
        print(f"[INFO] DataParallel on {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    model = model.to(device)
    return model, device

model, device = build_model()
USE_AMP = device.type == "cuda"
print(f"Device: {device}  |  Params: {sum(p.numel() for p in model.parameters()):,}")

## 6 │ Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    correct = total = 0
    for images, labels in tqdm(loader, desc="  train", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = total = 0
    all_logits, all_labels = [], []
    for images, labels in tqdm(loader, desc="  val  ", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        all_logits.append(outputs.float().cpu())
        all_labels.append(labels.cpu())
    return running_loss / total, correct / total, torch.cat(all_logits), torch.cat(all_labels)

In [ ]:
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    lr_now = optimizer.param_groups[0]["lr"]
    print(f"\nEpoch {epoch}/{EPOCHS}  (lr={lr_now:.2e})")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device
    )
    val_loss, val_acc, val_logits, val_labels = validate(
        model, val_loader, criterion, device
    )

    old_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]["lr"]
    if new_lr < old_lr:
        print(f"  ⚠ LR reduced: {old_lr:.2e} → {new_lr:.2e}")

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"  Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_path = os.path.join(OUTPUT_DIR, "model.pth")
        state = (model.module.state_dict()
                 if isinstance(model, nn.DataParallel)
                 else model.state_dict())
        torch.save(state, ckpt_path)
        print(f"  ✓ Best model saved → {ckpt_path}")

print("\n" + "=" * 50)
print("  Training complete!")
print("=" * 50)

### Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_range, history["train_loss"], label="Train", lw=2)
axes[0].plot(epochs_range, history["val_loss"],   label="Val",   lw=2)
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], label="Train", lw=2)
axes[1].plot(epochs_range, history["val_acc"],   label="Val",   lw=2)
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle("Training History", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 7 │ ROC-AUC Evaluation

In [ ]:
def evaluate_roc_auc(logits, labels):
    probs = torch.softmax(logits, dim=1).numpy()
    labels_np = labels.numpy()
    labels_bin = label_binarize(labels_np, classes=[0, 1, 2])

    macro_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")

    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ["#2196F3", "#FF9800", "#4CAF50"]
    for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
        fpr, tpr, _ = roc_curve(labels_bin[:, i], probs[:, i])
        ax.plot(fpr, tpr, color=color, lw=2,
                label=f"{cls_name} (AUC = {auc(fpr, tpr):.3f})")

    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random")
    ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate")
    ax.set_title(f"ROC Curves — Macro AUC = {macro_auc:.4f}", fontsize=14)
    ax.legend(loc="lower right", fontsize=11)
    ax.grid(alpha=0.3)
    fig.tight_layout()

    roc_path = os.path.join(OUTPUT_DIR, "roc_curve.png")
    fig.savefig(roc_path, dpi=150)
    plt.show()
    print(f"\nROC curve saved → {roc_path}")
    return macro_auc


macro_auc = evaluate_roc_auc(val_logits, val_labels)

print(f"\n{'='*50}")
print(f"  Macro ROC-AUC (OvR) : {macro_auc:.4f}")
print(f"  Model  : {os.path.join(OUTPUT_DIR, 'model.pth')}")
print(f"  ROC    : {os.path.join(OUTPUT_DIR, 'roc_curve.png')}")
print(f"{'='*50}")

## 8 │ Summary

| Component | Detail |
|---|---|
| **Backbone** | ResNet-18 (from scratch) |
| **Classes** | no → 0, sphere → 1, vort → 2 |
| **Augmentation** | 90° rotations, H-flip, V-flip |
| **Normalisation** | Min-max `[0, 1]` (no ImageNet stats) |
| **Optimizer** | Adam, lr=1e-4 |
| **Scheduler** | ReduceLROnPlateau (patience=5) |
| **Precision** | Mixed (FP16 via torch.amp) |
| **Multi-GPU** | DataParallel (auto-detected) |
| **Metric** | Macro OvR ROC-AUC |

### References
1. Varma et al. (2020) — [arXiv:2005.05353](https://arxiv.org/pdf/2005.05353)
2. Alexander et al. (2020) — [ApJ 893 15](https://doi.org/10.3847/1538-4357/ab7925)